# Part 6: Hardware, from checkpoint to silicon

The thesis of this project is a hardware claim: binary $\{-1,+1\}$ matmuls map to FPGA
LUT and carry logic instead of DSP blocks. No accuracy table can prove that.

Proving it takes a second pipeline. The trained checkpoint is rebuilt as a fixed-point
export graph, pushed through hls4ml into a Vitis HLS project, C-synthesized on the
`mulder` machine against a real device, and only then parsed into numbers. Every stage
has a gate, because the characteristic failure mode of hardware translation is not a
crash — it is a design that compiles cleanly, synthesizes cleanly, and computes the wrong
function.

## Why hardware gets its own graph

The graph that trained is not the graph that ships. The QAT model carries training
machinery everywhere: full-precision latent weights behind a straight-through estimator,
learnable-bitwidth bookkeeping, calibration taps. Hardware needs the opposite — a
self-contained graph in which every quantity is a literal fixed-point constant or a
fixed-point operation.

The export step rebuilds the model with three transformations.

1. **Weights are pinned to literal ±1.** The latent kernels are re-binarized with exactly
   the same absmean arithmetic used in training (same centering offset $\alpha$, same
   scale $\beta = \mathrm{mean}|W - \alpha|$, same bipolar sign that never emits zero), so
   the export reproduces the trained forward pass rather than approximating it.
2. **$\beta$ is restored in-graph.** The binarized weights are pure ±1, so the per-tensor
   scale has to live somewhere. On the current norm-free graphs there is no LayerNorm to
   absorb it, so every layer's $\beta$ becomes an explicit affine right after its own
   matmul.
3. **The activation grids are frozen** exactly as trained, read back verbatim from the
   trained quantizers and never re-calibrated at export time.

Transformations 2 and 3 share one lesson: trained models *saturate* some of their
activation grids on purpose, and carrying $\beta$ downstream algebraically with
generously re-derived "safe" grids destroys that learned nonlinearity. That export
computed a measurably different function and was caught by the fidelity gate on
2026-08-04 (the full story is in part 8).

The $\beta$ constants themselves are encoded by a chosen mode. `fx8` renders each $\beta$
as an 8-bit fixed-point constant. The `csd2` / `csd3` modes use canonical signed digit
encoding, the classic multiplierless trick: write the constant in digits from
$\{-1, 0, +1\}$ and the multiplication decomposes into one shift-and-add (or subtract)
per nonzero digit, with no multiplier anywhere. `csd2` caps the count at two nonzero
digits, `csd3` at three — trading approximation error against adder count, hence against
LUTs.

The whole ladder is measured per export and stored in `export_verify.json`.

In [ ]:
import json

ev = json.load(open("examples/export_verify_r14n8_w1a8s3.json"))
print(ev["config"], "seed", ev["seed"],
      "| binary layers (one beta affine each):", ev["n_bitlayers"])
print()
print(f"{'beta encoding':<14}{'corr(scores)':>14}{'corr(logits)':>14}{'argmax agree':>14}")
for mode in ("csd2", "csd3", "fx8", "exact"):
    r = ev["gate1_ladder"][mode]
    print(f"{mode:<14}{r['corr_scores']:>14.6f}{r['corr_logits']:>14.6f}"
          f"{r['argmax_agreement']:>14.4f}")

The selection rule was pre-registered: ship the **cheapest rung that clears 0.997 on both
articles** of the round. On this article `csd2` misses badly (0.9839) and `csd3` misses
narrowly (0.9969, just under the bar), so **`fx8`** at 0.9988 is the cheapest rung that
clears — chosen once and applied to every export of the round.

In [ ]:
g = ev["gate1"]
print("GATE 1 (export graph vs trained QAT model)")
print("  jets scored        :", g["n"])
print("  corr(softmax)      :", round(g["corr_scores"], 6))
print(f"  threshold          : {ev['gate1_threshold']}  (norm-free policy, 2026-08-04)")
print("  beta mode shipped  :", g["beta_mode"])
print("  verdict            :", "PASS" if ev["gate1_pass"] else "FAIL")

## The two fidelity gates

Quantization-semantics drift — a wrong rounding mode, a dropped $\beta$, a saturated
accumulator — does not crash anything. It compiles, it synthesizes, and it produces a
resource table for a network nobody trained. Numbers-first checks are the only defense,
so no synthesis time is ever spent on a design whose numerical fidelity has not been
measured.

- **GATE 1** compares the export graph against the trained QAT model: Pearson correlation
  of their softmax outputs on 4,096 real jets.
- **GATE 2** compares the export graph against hls4ml's bit-accurate C simulation of the
  generated firmware — same statistic, bar ≥ 0.997.

Together they localize any failure. GATE 1 failing means the fault is in our export
mathematics; GATE 2 failing means the fault is in the hls4ml translation of a graph
already proven faithful. For the synthesized round-14 model the record reads GATE 1 =
0.998816 and GATE 2 bit-exact (`bnjettag/results/r14/hls_r14.md`).

The nominal GATE 1 bar is ≥ 0.9999. For norm-free graphs it is 0.997, a policy decision
made on 2026-08-04 after the measured ceiling of an *exact*-encoding norm-free export came
in at 0.9991 on the 16-constituent article and 0.9993 on the 8-constituent one. The
lowered bar records the honest limit of that representation; it is not a silent pass.

The operational rule is blunt: never ship a tarball whose gate values you have not read.

## Reuse factor: the resource–latency dial

One parameter dominates the shape of every hardware number: the **reuse factor** (RF),
how many times a single physical multiply-accumulate unit is time-shared within a layer.

At RF = 1 the design is fully unrolled — initiation interval II = 1 (a new jet accepted
every clock), minimum cycles, maximum fabric. At RF = $R$ each unit does $R$ operations in
sequence: resources shrink by roughly $1/R$ while interval and latency grow. Two
operating points recur in this project's tables: RF = 1 as the max-parallelism
characterization point, and RF = 256 as the folded, device-fit point.

The rule follows directly: the same model at two reuse factors is two different designs,
so **no resource or latency number is ever quoted without its RF**.

![One real 32→64 binary layer synthesized four ways, all DSP = 0 and bit-exact: a fully parallel parity arm at 270,382 LUT, Four-Russians recoding at 2.1× less, fold-by-10 at 9.3× less, and fold plus recoding at 18.0× less — the folded arms running at II = 10 per jet against the parity arm's II = 1, the open engineering item their own results file names. Generated by `bnjettag/code/plots/make_lab_slides.py` from the Vitis HLS 2023.2 (xcvu13p) reports under `bnjettag/results/adder-graph/e1/`.](figures/folding_arms.png)

## Export and convert

One command fetches the checkpoint, builds the export graph, runs GATE 1, converts
through hls4ml, runs GATE 2, and writes the mulder-ready tarball.

In [ ]:
# runs on: laptop (hls4ml 1.3.0; no synthesis yet)
!KERAS_BACKEND=tensorflow python code/convert_final.py \
    --variant w1a8 --seed 3 --wandb-run r14-l1x3-n8-w1a8-s3 \
    --rf 1 --beta-mode fx8

## C-synthesis on mulder

C-synthesis is Vitis HLS (2023.2 throughout) compiling the generated C++ into
register-transfer-level hardware and reporting **estimated** resources (LUT, FF, DSP,
BRAM, URAM) and latency in cycles against the target device and clock: here the VU13P
(`xcvu13p-flga2577-2-e`) at 2.5 ns, i.e. 400 MHz.

Two boundaries on what these numbers mean are enforced in every document. C-synthesis is
*not* place-and-route, so the figures are labeled synthesis estimates, not a routed chip.
And only designs that have actually been through C-synthesis may be called
"synthesized" — facts visible at the convert stage (for example that the weights are
carried as `ap_uint<1>`, so a binary MAC structurally cannot bind to a DSP) are quoted as
structural facts, never dressed up as measurements.

In [ ]:
# runs on: laptop -> mulder (Vitis HLS 2023.2)
!scp hls_prj_rf1.tar.gz mulder:~/bnjet_r14/
!ssh mulder 'cd ~/bnjet_r14 && ./mulder_csynth.sh hls_prj_rf1.tar.gz'

The driver script is deliberately boring. It sources the *full* Vitis `settings64.sh`
(which provides both `vitis-run` and `vitis_hls`), preserves any reports from a previous
run of the same name before touching anything, and writes the stage flags into
`build_opt.tcl` — the file the hls4ml build script actually reads.

In [ ]:
sh = open("code/mulder_csynth.sh").read().splitlines()
i = next(k for k, l in enumerate(sh) if l.startswith("  # Synthesis only."))
print("\n".join(sh[i:i + 16]))

That block is a scar. Passing the stage flags as a string to `vitis_hls -f` was silently
ignored, so for a while every "synthesis" run also did C-simulation, Verilog cosimulation
and validation, for hours — found on 2026-08-04 (part 8).

The script then starts a memory watch, because large fully-unrolled designs die by
SIGKILL around 112 GB of resident memory on the 125 GB machine. The watch turns an
out-of-memory death from a mystery into a diagnosis: it is how the round-14 $N = 16$
attempt, dead after 17.7 hours with no report, is known to be the large-design pathology
rather than anything about the model (`bnjettag/results/r14/hls_r14.md`).

Finally it runs `vitis_hls` and pipes the resulting `csynth.xml` through
`parse_csynth.py`, a parser kept standard-library-only on purpose so it runs on mulder's
bare system Python.

## Two measured operating points

The round-14 $N = 8$ W1A8 model, whole-model synthesis at **RF = 1**, Latency strategy,
was measured twice: once with multiplies left unconstrained (`fx8`) and once with
multiplies forced into fabric (`config_op mul -impl fabric`).

In [ ]:
import json

rep = {m: json.load(open(f"examples/csynth_report_r14n8_{m}_rf1.json"))
       for m in ("fx8", "fabric")}

def pct(r, k):
    return f"{r[k]:,} ({100 * r[k] / r['avail'][k]:.1f}%)"

metrics = [
    ("DSP",                  lambda r: f"{r['DSP']:,}"),
    ("LUT (% of device)",    lambda r: pct(r, "LUT")),
    ("FF (% of device)",     lambda r: pct(r, "FF")),
    ("latency (cycles)",     lambda r: str(r["LatencyBest"])),
    ("initiation interval",  lambda r: str(r["IntervalMin"])),
    ("estimated clock (ns)", lambda r: r["estimated_clock_ns"]),
    ("target clock (ns)",    lambda r: r["target_clock_ns"]),
]

print(f"{'':22}{'fx8':>22}{'fabric':>22}")
for name, f in metrics:
    print(f"{name:22}{f(rep['fx8']):>22}{f(rep['fabric']):>22}")

r = rep["fx8"]
print(f"\nfx8 latency at the ACHIEVED clock: {r['LatencyBest']} cycles x "
      f"{r['estimated_clock_ns']} ns = {r['LatencyBest'] * float(r['estimated_clock_ns']):.0f} ns")
print("both points: RF = 1, whole model, Latency strategy, part", r["part"])

That table is the DSP↔LUT lever, measured. Forcing multiplies into fabric takes the
design from 4,133 DSPs to **zero in all 106 instances** — the first whole-model synthesis
of the stack with no DSPs anywhere — and pays for it in LUTs, 3,178,720 → 4,376,222
(184.0% → 253.3% of the VU13P; RF = 1).

The `fx8` point shows where DSPs go when they are permitted: of its 4,133, some 3,621 sit
in the fifteen $\beta$-restoration affines and 512 in softmax, while **every binary matmul
and all four activation×activation attention einsums use exactly zero**
(`bnjettag/results/r14/hls_r14.md`).

Two honest caveats travel with those numbers.

The over-100% LUT figures are expected and labeled: RF = 1 is the max-parallelism
reference point, not a deployment proposal — this design does not fit the device, and the
folded, deployable-class operating points are a separate pre-registered table that is
never merged into the same column. And the fabric point's estimated clock is below
target, carrying Vitis timing-violation flags, so no nanosecond latency is quoted for it;
the `fx8` point's 299 ns is quoted at its achieved 1.825 ns clock.

![The accumulator lever: identical binary weights synthesize to 270 DSP blocks with the default narrow accumulator and to 14 (all in the packaged SubLN norm; the matmul itself zero) with a widened accumulator, for the attention Q/K/V and W_o dense layers at RF = 256, Resource strategy. Measured pairs from `bnjettag/results/final/hls_resource_table_final.md`.](figures/dsp_accumulator_lever.png)

That figure is the backstory to check 1 below: "binary weights ⇒ 0 DSP" is not automatic,
it has an enabling condition. With the toolflow's minimal accumulator a real binary
attention matmul synthesized to 270 DSPs (RF = 256, Resource strategy); widening the
accumulator — mathematically exact, since it only adds precision — dropped the same layer
to 14 DSPs, all of them in the norm packaged with the probe and none in the matmul, for
19–24k LUT at RF = 256.

## The standard checks

Every report that comes home passes the same five checks, each with a reason.

1. **DSP = 0 on every binary matmul.** The thesis check, run before anything else. Any
   DSP attributed to a binary layer means the tool inferred a genuine multiply where the
   claim says add/subtract, and nothing else in the report matters until that is explained.
2. **Estimated clock ≤ 2.5 ns** (estimated Fmax ≥ 400 MHz). Below target, the timing claim
   fails even as an estimate.
3. **Latency in microseconds at the *achieved* clock** (cycles × estimated ns), never at
   the target clock, which flatters the number whenever the estimate beats target.
4. **Percentages against the report's own `avail{}` block**, never against device totals
   recalled from memory — which is exactly what the comparison cell above does.
5. **Fidelity metrics recorded beside the hardware numbers.** A resource table for a
   design whose gates failed is a table about nothing, so GATE 1 and GATE 2 values sit in
   the same file as the LUT counts they license.

![Tree-exact per-module LUT census of the whole-model round-8 flagship (norm-free) at RF = 8 (II = 8; 3,910,515 LUT total): per-token binary dense slices dominate at 61.9%, attention act×act einsums second at 20.2%; the binary dense share alone exceeds the VU13P budget while everything else fits. Generated by `bnjettag/code/plots/make_lab_slides.py` from `bnjettag/r7/results/csynth/whole_model_rf8_stdnn.xml`.](figures/lut_budget.png)

## Bringing the numbers home

The fetch script closes the loop with one rule baked into it: the JSON report is
regenerated **on mulder, from the raw `csynth.xml`**, before being copied home together
with that XML.

In [ ]:
# runs on: laptop (pulls from mulder, regenerating each report from its XML)
!./code/fetch_mulder_reports.sh

A `csynth_report.json` that rode *to* mulder inside the tarball is never a result — it is
whatever the laptop believed before synthesis ran — so the fetch script deletes such files
rather than copy them. The raw XML always lands beside the parsed row, so every table cell
can be re-derived from the primary artifact, and per-module attribution
(`parse_csynth_modules.py`) answers the question every DSP count raises: which module,
exactly, is using them.

Record-keeping differs from the accuracy side in one respect. The parsed local tables, not
W&B, are the hardware numbers of record; W&B holds a provenance copy of each report linked
back to its training run. And the independent verification pass of part 5 applies here in
kind: every cell of the round-14 table was re-derived from the raw XML and report files
before the table was allowed to say "numbers of record"
(`bnjettag/results/r14/verification_hls_r14_2026-08-07.txt`).